In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# df1 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260211.parquet")
# df2 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260212.parquet")
# df3 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260213.parquet")
# df4 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260214.parquet")
# df5 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260215.parquet")
# df6 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260216.parquet")
# df7 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260217.parquet")
# df8 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260218.parquet")
# df9 = pd.read_parquet("C:/msys64/home/for/10th/00_Project/04_final/02_parquet_file/Erangel_telemetry_kakao_20260219.parquet")


from pathlib import Path
import re
import pandas as pd
import pyarrow.dataset as ds

BASE = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file")  # 내 경로로 수정
MAP = "Erangel"
PLATFORM = "kakao"
DATE_FROM = "20260211"
DATE_TO   = "20260219"

# 1) 대상 parquet 파일만 골라오기
pat = re.compile(rf"^{MAP}_telemetry_{PLATFORM}_(\d{{8}})\.parquet$")
files = []
for fp in sorted(BASE.glob(f"{MAP}_telemetry_{PLATFORM}_*.parquet")):
    m = pat.match(fp.name)
    if not m:
        continue
    yyyymmdd = m.group(1)
    if DATE_FROM <= yyyymmdd <= DATE_TO:
        files.append(fp)

print("files:", len(files))
if not files:
    raise SystemExit("No parquet files matched. 패턴/날짜/경로 확인 필요")

# 2) 출력 폴더 준비
OUTDIR = BASE / f"csv_{MAP}_{PLATFORM}_{DATE_FROM}_{DATE_TO}"
OUTDIR.mkdir(parents=True, exist_ok=True)

EVENTS = [
    "LogPlayerAttack",
    "LogPlayerMakeGroggy",
    "LogPlayerTakeDamage",
    "LogPlayerKillV2",
    "LogPlayerPosition",
    "LogItemPickup",
    "LogMatchStart",
    "LogMatchEnd",
    "LogPhaseChange",
]

# 3) 스키마 기반으로 실제 존재 컬럼만 쓰도록 처리(중요: 컬럼명 불일치로 에러 방지)
dataset = ds.dataset([str(p) for p in files], format="parquet")
all_cols = set(dataset.schema.names)

# 최소 공통 컬럼 (너 데이터에 맞게 필요하면 추가)
BASE_COLS = ["_D", "_T", "matchId"]  # accountId가 있다면 추가 가능
use_cols = [c for c in BASE_COLS if c in all_cols]

# 너무 많이 읽고 싶으면 아래처럼 "필요한 컬럼"을 더 추가해도 됨 (존재하는 것만)
CANDIDATES = [
    "accountId", "playerId", "attacker", "victim", "damage", "damageDealt",
    "killer", "killed", "weapon", "item", "itemId", "itemName",
    "x", "y", "z", "location", "character", "teamId",
    "time", "timestamp"
]
use_cols += [c for c in CANDIDATES if c in all_cols]
use_cols = list(dict.fromkeys(use_cols))  # 중복 제거, 순서 유지

print("use_cols:", use_cols)

# 4) 배치로 스캔 (전체 1-pass) 후 _T로 분리 저장
scanner = dataset.scanner(columns=use_cols, batch_size=65536)

# 헤더 1회만 쓰기 위한 플래그
written = {ev: False for ev in EVENTS}

for batch in scanner.to_batches():
    df = batch.to_pandas()

    # _T 컬럼이 없으면 작업 불가
    if "_T" not in df.columns:
        raise RuntimeError("_T column not found in loaded batch. 스키마/컬럼명 확인 필요")

    # 관심 이벤트만 필터
    df = df[df["_T"].isin(EVENTS)]
    if df.empty:
        continue

    # 이벤트별로 쪼개서 append 저장
    for ev, sub in df.groupby("_T"):
        out = OUTDIR / f"{MAP}_{PLATFORM}_{DATE_FROM}_{DATE_TO}__{ev}.csv"
        sub.to_csv(out, mode="a", index=False, header=(not written[ev]))
        written[ev] = True

print("done. output dir:", OUTDIR)
print("written:", {k:v for k,v in written.items() if v})


files: 9
use_cols: ['_D', '_T', 'matchId', 'accountId', 'attacker', 'damage', 'killer']


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import pyarrow.dataset as ds
import pandas as pd

# ===== 설정 =====
PARQUET_PATH = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\Erangel_telemetry_kakao_20260211.parquet")
EVENT_NAME = "LogPlayerAttack"   # 예: "LogPlayerTakeDamage", "LogPlayerKillV2" 등
BATCH_SIZE = 65536

# ===== 로드 (pyarrow.dataset) =====
dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
all_cols = dataset.schema.names

# _T 컬럼이 반드시 있어야 이벤트 필터 가능
if "_T" not in all_cols:
    raise RuntimeError("이 parquet에 _T 컬럼이 없습니다. 컬럼명 확인 필요.")

# 결과 누적용: 각 컬럼이 'non-null 값이 한 번이라도 있었는지'
has_value = {c: False for c in all_cols}
rows_seen = 0
rows_matched = 0

scanner = dataset.scanner(columns=all_cols, batch_size=BATCH_SIZE)

for batch in scanner.to_batches():
    df = batch.to_pandas()
    rows_seen += len(df)

    # 이벤트 필터
    sub = df[df["_T"] == EVENT_NAME]
    if sub.empty:
        continue

    rows_matched += len(sub)

    # 컬럼별로 non-null 존재 여부 체크
    # (이미 True 된 컬럼은 스킵해서 속도 조금 개선)
    for c in all_cols:
        if has_value[c]:
            continue
        # pandas 기준: NaN/None 모두 notna()에서 False 처리
        if sub[c].notna().any():
            has_value[c] = True

print(f"전체 행: {rows_seen:,}")
print(f"{EVENT_NAME} 행: {rows_matched:,}")

non_null_cols = [c for c, v in has_value.items() if v]
print(f"\n[{EVENT_NAME}]에서 값이 존재하는 컬럼 수: {len(non_null_cols)}")
print(non_null_cols)


전체 행: 4,417,623
LogPlayerAttack 행: 664,675

[LogPlayerAttack]에서 값이 존재하는 컬럼 수: 42
['matchId', '_D', '_T', 'common_isGame', 'attackId', 'fireWeaponStackCount', 'attacker_name', 'attacker_teamId', 'attacker_health', 'attacker_location_x', 'attacker_location_y', 'attacker_location_z', 'attacker_ranking', 'attacker_individualRanking', 'attacker_accountId', 'attacker_isInBlueZone', 'attacker_isInRedZone', 'attacker_inSpecialZone', 'attacker_isInVehicle', 'attacker_zone', 'attacker_type', 'attacker_isDBNO', 'attackType', 'weapon_itemId', 'weapon_stackCount', 'weapon_category', 'weapon_subCategory', 'weapon_attachedItems', 'vehicle_vehicleType', 'vehicle_vehicleId', 'vehicle_seatIndex', 'vehicle_healthPercent', 'vehicle_feulPercent', 'vehicle_altitudeAbs', 'vehicle_altitudeRel', 'vehicle_velocity', 'vehicle_isWheelsInAir', 'vehicle_isInWaterVolume', 'vehicle_isEngineOn', 'vehicle_location_x', 'vehicle_location_y', 'vehicle_location_z']


In [ ]:
from pathlib import Path
import pyarrow.dataset as ds
import pandas as pd

PARQUET_PATH = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\Erangel_telemetry_kakao_20260211.parquet")
BATCH_SIZE = 65536

EVENTS = [
    "LogParachuteLanding",
    "LogPlayerPosition",
    "LogMatchStart",
    "LogGameStatePeriodic",
    "LogVehicleRide",
    "LogVehicleLeave",
    "LogPhaseChange"
]

dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
all_cols = dataset.schema.names

if "_T" not in all_cols:
    raise RuntimeError("이 parquet에 _T 컬럼이 없습니다. 컬럼명 확인 필요.")

# 이벤트별로: 컬럼이 한 번이라도 non-null이었는지 저장
has_value = {ev: {c: False for c in all_cols} for ev in EVENTS}
rows_seen = 0
rows_matched = {ev: 0 for ev in EVENTS}

scanner = dataset.scanner(columns=all_cols, batch_size=BATCH_SIZE)

for batch in scanner.to_batches():
    df = batch.to_pandas()
    rows_seen += len(df)

    # 관심 이벤트만 남기기
    df = df[df["_T"].isin(EVENTS)]
    if df.empty:
        continue

    # 이벤트별로 처리
    for ev, sub in df.groupby("_T"):
        rows_matched[ev] += len(sub)

        # 이미 True인 컬럼은 스킵
        hv = has_value[ev]
        for c in all_cols:
            if hv[c]:
                continue
            if sub[c].notna().any():
                hv[c] = True

print(f"전체 행: {rows_seen:,}")
print("========================================")

for ev in EVENTS:
    non_null_cols = [c for c, v in has_value[ev].items() if v]
    print(f"\n[{ev}]")
    print(f"  행 수: {rows_matched[ev]:,}")
    print(f"  값이 존재하는 컬럼 수: {len(non_null_cols)}")
    print(f"  컬럼: {non_null_cols}")


전체 행: 4,417,623

[LogParachuteLanding]
  행 수: 10,170
  값이 존재하는 컬럼 수: 21
  컬럼: ['matchId', '_D', '_T', 'common_isGame', 'character_name', 'character_teamId', 'character_health', 'character_location_x', 'character_location_y', 'character_location_z', 'character_ranking', 'character_individualRanking', 'character_accountId', 'character_isInBlueZone', 'character_isInRedZone', 'character_inSpecialZone', 'character_isInVehicle', 'character_zone', 'character_type', 'character_isDBNO', 'distance']

[LogPlayerPosition]
  행 수: 773,989
  값이 존재하는 컬럼 수: 36
  컬럼: ['matchId', '_D', '_T', 'common_isGame', 'character_name', 'character_teamId', 'character_health', 'character_location_x', 'character_location_y', 'character_location_z', 'character_ranking', 'character_individualRanking', 'character_accountId', 'character_isInBlueZone', 'character_isInRedZone', 'character_inSpecialZone', 'character_isInVehicle', 'character_zone', 'character_type', 'character_isDBNO', 'elapsedTime', 'numAlivePlayers', 'vehi

In [ ]:
from pathlib import Path
import pyarrow.dataset as ds
import pandas as pd

PARQUET_PATH = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\Erangel_telemetry_kakao_20260211.parquet")
BATCH_SIZE = 65536

EVENTS = [
    "LogParachuteLanding",
    "LogPlayerPosition",
    "LogMatchStart",
    "LogGameStatePeriodic",
    "LogVehicleRide",
    "LogVehicleLeave",
    "LogPhaseChange"
]

dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
all_cols = dataset.schema.names

if "_T" not in all_cols:
    raise RuntimeError("이 parquet에 _T 컬럼이 없습니다. 컬럼명 확인 필요.")

# =========================
# PASS 1) 이벤트별 non-null 컬럼 찾기
# =========================
has_value = {ev: {c: False for c in all_cols} for ev in EVENTS}
rows_matched = {ev: 0 for ev in EVENTS}
rows_seen = 0

scanner = dataset.scanner(columns=all_cols, batch_size=BATCH_SIZE)

for batch in scanner.to_batches():
    df = batch.to_pandas()
    rows_seen += len(df)

    df = df[df["_T"].isin(EVENTS)]
    if df.empty:
        continue

    for ev, sub in df.groupby("_T"):
        rows_matched[ev] += len(sub)
        hv = has_value[ev]
        for c in all_cols:
            if hv[c]:
                continue
            if sub[c].notna().any():
                hv[c] = True

non_null_cols_by_event = {}
for ev in EVENTS:
    cols = [c for c, v in has_value[ev].items() if v]
    # 안전장치: 최소한 핵심은 항상 포함
    for must in ["matchId", "_D", "_T"]:
        if must in all_cols and must not in cols:
            cols.insert(0, must)
    non_null_cols_by_event[ev] = cols

print(f"PASS1 done. total rows seen: {rows_seen:,}")
for ev in EVENTS:
    print(f"{ev:20s} rows={rows_matched[ev]:,} non_null_cols={len(non_null_cols_by_event[ev])}")

# =========================
# PASS 2) non-null 컬럼만 골라서 이벤트별 CSV 저장
# =========================
OUTDIR = PARQUET_PATH.parent / f"csv_nonnull_{PARQUET_PATH.stem}"
OUTDIR.mkdir(parents=True, exist_ok=True)

out_path = {ev: OUTDIR / f"{ev}.csv" for ev in EVENTS}

# 기존 파일 삭제(덮어쓰기)
for ev, p in out_path.items():
    if p.exists():
        p.unlink()

written_header = {ev: False for ev in EVENTS}
rows_written = {ev: 0 for ev in EVENTS}

# (중요) PASS2에서는 "이벤트별 필요한 컬럼들의 합집합"만 읽어서 I/O 줄이기
union_cols = set()
for ev in EVENTS:
    union_cols.update(non_null_cols_by_event[ev])
union_cols = [c for c in all_cols if c in union_cols]  # schema 순서 유지

scanner2 = dataset.scanner(columns=union_cols, batch_size=BATCH_SIZE)

for batch in scanner2.to_batches():
    df = batch.to_pandas()
    df = df[df["_T"].isin(EVENTS)]
    if df.empty:
        continue

    for ev, sub in df.groupby("_T"):
        cols = non_null_cols_by_event[ev]
        sub = sub[cols]  # 여기서 non-null 컬럼만 남김

        sub.to_csv(
            out_path[ev],
            mode="a",
            index=False,
            header=(not written_header[ev])
        )
        written_header[ev] = True
        rows_written[ev] += len(sub)

print("\nPASS2 done. outputs:")
for ev in EVENTS:
    print(f"{ev:20s} -> {rows_written[ev]:,} rows | {out_path[ev]}")

PASS1 done. total rows seen: 4,417,623
LogParachuteLanding  rows=10,170 non_null_cols=21
LogPlayerPosition    rows=773,989 non_null_cols=36
LogMatchStart        rows=118 non_null_cols=12
LogGameStatePeriodic rows=19,687 non_null_cols=28
LogVehicleRide       rows=78,204 non_null_cols=36
LogVehicleLeave      rows=78,180 non_null_cols=38
LogPhaseChange       rows=1,761 non_null_cols=6

PASS2 done. outputs:
LogParachuteLanding  -> 10,170 rows | C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\csv_nonnull_Erangel_telemetry_kakao_20260211\LogParachuteLanding.csv
LogPlayerPosition    -> 773,989 rows | C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\csv_nonnull_Erangel_telemetry_kakao_20260211\LogPlayerPosition.csv
LogMatchStart        -> 118 rows | C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\csv_nonnull_Erangel_telemetry_kakao_20260211\LogMatchStart.csv
LogGameStatePeriodic -> 19,687 rows | C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\csv_

In [3]:
from pathlib import Path
import re
import pyarrow.dataset as ds
import pandas as pd

# ===== 설정 =====
BASE_DIR = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file")
BATCH_SIZE = 65536

EVENTS = [
    "LogParachuteLanding",
    "LogPlayerPosition",
    "LogMatchStart",
    "LogGameStatePeriodic",
    "LogVehicleRide",
    "LogVehicleLeave",
    "LogPhaseChange",
]

# 너가 이미 출력으로 확인한 "값이 존재하는 컬럼" 리스트 (Erangel_kakao_20260211 기준)
# (대부분 맵/플랫폼에서 스키마가 동일해서 그대로 써도 무방)
NON_NULL_COLS = {
    "LogParachuteLanding": ['matchId','_D','_T','common_isGame','character_name','character_teamId','character_health',
                            'character_location_x','character_location_y','character_location_z','character_ranking',
                            'character_individualRanking','character_accountId','character_isInBlueZone','character_isInRedZone',
                            'character_inSpecialZone','character_isInVehicle','character_zone','character_type','character_isDBNO','distance'],

    "LogPlayerPosition": ['matchId','_D','_T','common_isGame','character_name','character_teamId','character_health',
                          'character_location_x','character_location_y','character_location_z','character_ranking',
                          'character_individualRanking','character_accountId','character_isInBlueZone','character_isInRedZone',
                          'character_inSpecialZone','character_isInVehicle','character_zone','character_type','character_isDBNO',
                          'elapsedTime','numAlivePlayers','vehicle_vehicleType','vehicle_vehicleId','vehicle_seatIndex',
                          'vehicle_healthPercent','vehicle_feulPercent','vehicle_altitudeAbs','vehicle_altitudeRel','vehicle_velocity',
                          'vehicle_isWheelsInAir','vehicle_isInWaterVolume','vehicle_isEngineOn','vehicle_location_x','vehicle_location_y','vehicle_location_z'],

    "LogMatchStart": ['matchId','_D','_T','common_isGame','mapName','weatherId','characters','cameraViewBehaviour',
                      'teamSize','isCustomGame','isEventMode','blueZoneCustomOptions'],

    "LogGameStatePeriodic": ['matchId','_D','_T','common_isGame','gameState_elapsedTime','gameState_numStartTeams',
                             'gameState_numAliveTeams','gameState_numParticipatedTeams','gameState_numJoinPlayers',
                             'gameState_numStartPlayers','gameState_numAlivePlayers','gameState_numParticipatedPlayers',
                             'gameState_safetyZonePosition_x','gameState_safetyZonePosition_y','gameState_safetyZonePosition_z',
                             'gameState_safetyZoneRadius','gameState_poisonGasWarningPosition_x','gameState_poisonGasWarningPosition_y',
                             'gameState_poisonGasWarningPosition_z','gameState_poisonGasWarningRadius','gameState_redZonePosition_x',
                             'gameState_redZonePosition_y','gameState_redZonePosition_z','gameState_redZoneRadius',
                             'gameState_blackZonePosition_x','gameState_blackZonePosition_y','gameState_blackZonePosition_z',
                             'gameState_blackZoneRadius'],

    "LogVehicleRide": ['matchId','_D','_T','common_isGame','character_name','character_teamId','character_health',
                       'character_location_x','character_location_y','character_location_z','character_ranking',
                       'character_individualRanking','character_accountId','character_isInBlueZone','character_isInRedZone',
                       'character_inSpecialZone','character_isInVehicle','character_zone','character_type','character_isDBNO',
                       'vehicle_vehicleType','vehicle_vehicleId','vehicle_seatIndex','vehicle_healthPercent','vehicle_feulPercent',
                       'vehicle_altitudeAbs','vehicle_altitudeRel','vehicle_velocity','vehicle_isWheelsInAir','vehicle_isInWaterVolume',
                       'vehicle_isEngineOn','vehicle_location_x','vehicle_location_y','vehicle_location_z','seatIndex','fellowPassengers'],

    "LogVehicleLeave": ['matchId','_D','_T','common_isGame','character_name','character_teamId','character_health',
                        'character_location_x','character_location_y','character_location_z','character_ranking',
                        'character_individualRanking','character_accountId','character_isInBlueZone','character_isInRedZone',
                        'character_inSpecialZone','character_isInVehicle','character_zone','character_type','character_isDBNO',
                        'vehicle_vehicleType','vehicle_vehicleId','vehicle_seatIndex','vehicle_healthPercent','vehicle_feulPercent',
                        'vehicle_altitudeAbs','vehicle_altitudeRel','vehicle_velocity','vehicle_isWheelsInAir','vehicle_isInWaterVolume',
                        'vehicle_isEngineOn','vehicle_location_x','vehicle_location_y','vehicle_location_z','seatIndex',
                        'fellowPassengers','rideDistance','maxSpeed'],

    "LogPhaseChange": ['matchId','_D','_T','common_isGame','phase','playersInWhiteCircle'],
}

# 출력 폴더
OUT_BASE = BASE_DIR / "03_mapwise_logs_kakao+steam_20260211_20260219"
OUT_BASE.mkdir(parents=True, exist_ok=True)

# 파일명 파서: {Map}_telemetry_{platform}_{YYYYMMDD}.parquet
pat = re.compile(r"^(?P<map>[^_]+)_telemetry_(?P<platform>kakao|steam)_(?P<date>\d{8})\.parquet$", re.IGNORECASE)

# 대상 parquet 수집
parquets = []
for p in BASE_DIR.glob("*.parquet"):
    m = pat.match(p.name)
    if not m:
        continue
    info = m.groupdict()
    info["path"] = p
    parquets.append(info)

# 날짜 범위(원하면 여기서 필터 가능)
DATE_MIN = "20260211"
DATE_MAX = "20260219"
parquets = [x for x in parquets if DATE_MIN <= x["date"] <= DATE_MAX]

if not parquets:
    raise RuntimeError("조건에 맞는 parquet가 없습니다. 파일명/경로/날짜 범위를 확인하세요.")

# 이벤트별로 필요한 컬럼 합집합만 읽어서 I/O 줄이기
union_cols = set(["_T"])  # _T는 필수
for ev in EVENTS:
    union_cols.update(NON_NULL_COLS[ev])
union_cols = list(union_cols)

# 맵/이벤트별 헤더 1회만 쓰기 위한 플래그
written_header = {}  # key: (map, ev) -> bool
rows_written = {}    # key: (map, ev) -> int

def out_csv_path(map_name: str, ev: str) -> Path:
    d = OUT_BASE / map_name
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{ev}.csv"

for f in parquets:
    map_name = f["map"]
    platform = f["platform"].lower()
    date = f["date"]
    path = f["path"]

    print(f"Processing: {path.name}")

    dataset = ds.dataset(str(path), format="parquet")
    available_cols = set(dataset.schema.names)

    if "_T" not in available_cols:
        print(f"  SKIP: _T 없음 -> {path.name}")
        continue

    # 이 파일에서 존재하는 컬럼만 읽기
    cols_to_read = [c for c in union_cols if c in available_cols]

    # arrow 레벨에서 이벤트 필터 (불필요한 행을 pandas로 넘기지 않음)
    filt = ds.field("_T").isin(EVENTS)

    scanner = dataset.scanner(columns=cols_to_read, filter=filt, batch_size=BATCH_SIZE)

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        if df.empty:
            continue

        # 이벤트별로 분리 저장
        for ev, sub in df.groupby("_T"):
            # 이 이벤트에서 “값 있는 컬럼”만 남기기(단, 파일에 존재하는 컬럼만)
            cols = [c for c in NON_NULL_COLS[ev] if c in sub.columns]
            sub = sub[cols].copy()

            # 출처 메타 추가(나중에 정렬/필터 편하게)
            sub["src_map"] = map_name
            sub["src_platform"] = platform
            sub["src_date"] = date

            outp = out_csv_path(map_name, ev)
            key = (map_name, ev)

            header = not written_header.get(key, False)
            sub.to_csv(outp, mode="a", index=False, header=header)
            written_header[key] = True
            rows_written[key] = rows_written.get(key, 0) + len(sub)

print("\n==== DONE ====")
# 결과 요약 출력
for map_name in sorted(set(x["map"] for x in parquets)):
    print(f"\n[{map_name}]")
    for ev in EVENTS:
        n = rows_written.get((map_name, ev), 0)
        print(f"  {ev:20s}: {n:,} rows  ->  {OUT_BASE / map_name / (ev + '.csv')}")

Processing: Erangel_telemetry_kakao_20260211.parquet
Processing: Erangel_telemetry_kakao_20260212.parquet
Processing: Erangel_telemetry_kakao_20260213.parquet
Processing: Erangel_telemetry_kakao_20260214.parquet
Processing: Erangel_telemetry_kakao_20260215.parquet
Processing: Erangel_telemetry_kakao_20260216.parquet
Processing: Erangel_telemetry_kakao_20260217.parquet
Processing: Erangel_telemetry_kakao_20260218.parquet
Processing: Erangel_telemetry_kakao_20260219.parquet
Processing: Erangel_telemetry_steam_20260211.parquet
Processing: Erangel_telemetry_steam_20260212.parquet
Processing: Erangel_telemetry_steam_20260213.parquet
Processing: Erangel_telemetry_steam_20260214.parquet
Processing: Erangel_telemetry_steam_20260215.parquet
Processing: Erangel_telemetry_steam_20260216.parquet
Processing: Erangel_telemetry_steam_20260217.parquet
Processing: Erangel_telemetry_steam_20260218.parquet
Processing: Erangel_telemetry_steam_20260219.parquet
Processing: Miramar_telemetry_kakao_20260211.p